# 📦 Data Preparation — NSL-KDD
Loads the raw NSL-KDD dataset, maps 39 specific attack types to 5 broad classes,
applies one-hot encoding to categorical features, aligns train/test columns,
and saves the processed splits to `data/processed/`.

> **No scaling or balancing here** — those are training concerns handled in
> `train_model.ipynb` to prevent data leakage.

**Run order:** this notebook → `train_model.ipynb`


## Imports

In [7]:
import os
import pandas as pd
import joblib

## Constants

In [8]:
RAW_COLUMNS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

CATEGORICAL_COLS = ['protocol_type', 'service', 'flag']

# Maps every specific attack name to one of 5 broad categories
ATTACK_MAPPING = {
    # DoS — Denial of Service
    'neptune': 'DoS', 'smurf': 'DoS', 'back': 'DoS', 'teardrop': 'DoS',
    'pod': 'DoS', 'land': 'DoS', 'apache2': 'DoS', 'udpstorm': 'DoS',
    'processtable': 'DoS', 'mailbomb': 'DoS', 'worm': 'DoS',
    # Probe — Surveillance / scanning
    'satan': 'Probe', 'ipsweep': 'Probe', 'portsweep': 'Probe', 'nmap': 'Probe',
    'mscan': 'Probe', 'saint': 'Probe',
    # R2L — Remote to Local
    'warezclient': 'R2L', 'guess_passwd': 'R2L', 'warezmaster': 'R2L',
    'imap': 'R2L', 'ftp_write': 'R2L', 'multihop': 'R2L', 'phf': 'R2L',
    'spy': 'R2L', 'snmpgetattack': 'R2L', 'snmpguess': 'R2L', 'sendmail': 'R2L',
    'named': 'R2L', 'xlock': 'R2L', 'xsnoop': 'R2L', 'httptunnel': 'R2L',
    # U2R — User to Root (privilege escalation)
    'buffer_overflow': 'U2R', 'rootkit': 'U2R', 'loadmodule': 'U2R',
    'perl': 'U2R', 'ps': 'U2R', 'xterm': 'U2R', 'sqlattack': 'U2R',
    # Normal
    'normal': 'Normal',
}

CLASS_MAPPING = {'Normal': 0, 'DoS': 1, 'Probe': 2, 'R2L': 3, 'U2R': 4}

TRAIN_PATH = os.path.join("..",'data', 'KDDTrain+.txt')
TEST_PATH  = os.path.join("..",'data', 'KDDTest+.txt')
OUT_DIR    = os.path.join("..",'data', 'processed')

## Step 1 — Load Raw Data

In [9]:
def load_and_clean(path):
    """Read a raw NSL-KDD .txt file and drop the unused difficulty_level column."""
    df = pd.read_csv(path, names=RAW_COLUMNS)
    df = df.drop(columns=['difficulty_level'])
    return df

train_df = load_and_clean(TRAIN_PATH)
test_df  = load_and_clean(TEST_PATH)

print(f"Train : {train_df.shape[0]:,} rows × {train_df.shape[1]} cols")
print(f"Test  : {test_df.shape[0]:,}  rows × {test_df.shape[1]} cols")
train_df.head(3)

Train : 125,973 rows × 42 cols
Test  : 22,544  rows × 42 cols


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,25,0.17,0.03,0.17,0.0,0.0,0.0,0.05,0.0,normal
1,0,udp,other,SF,146,0,0,0,0,0,...,1,0.00,0.60,0.88,0.0,0.0,0.0,0.00,0.0,normal
2,0,tcp,private,S0,0,0,0,0,0,0,...,26,0.10,0.05,0.00,0.0,1.0,1.0,0.00,0.0,neptune


## Step 2 — Map Attack Labels to 5 Classes

In [10]:
def map_labels(df):
    """Replace raw attack names with 5-class string labels."""
    df = df.copy()
    df['attack_class'] = df['label'].map(ATTACK_MAPPING).fillna('Normal')
    df = df.drop(columns=['label'])
    return df

train_df = map_labels(train_df)
test_df  = map_labels(test_df)

print("Train class distribution:\n")
total = len(train_df)
for cls, count in train_df['attack_class'].value_counts().items():
    bar = '█' * int(count / total * 40)
    print(f"  {cls:<8} {count:>7,}  ({count/total*100:5.2f}%)  {bar}")

Train class distribution:

  Normal    67,343  (53.46%)  █████████████████████
  DoS       45,927  (36.46%)  ██████████████
  Probe     11,656  ( 9.25%)  ███
  R2L          995  ( 0.79%)  
  U2R           52  ( 0.04%)  


## Step 3 — One-Hot Encoding & Column Alignment

`protocol_type`, `service`, and `flag` are categorical — we one-hot encode them.
The test set may have categories not seen in training (and vice versa), so we align
the test columns to the training columns, filling any missing ones with 0.


In [11]:
def encode_and_align(train_df, test_df):
    """One-hot encode categoricals and align test columns to training columns."""
    X_train_raw = train_df.drop(columns=['attack_class'])
    X_test_raw  = test_df.drop(columns=['attack_class'])

    y_train = train_df['attack_class'].map(CLASS_MAPPING)
    y_test  = test_df['attack_class'].map(CLASS_MAPPING)

    X_train = pd.get_dummies(X_train_raw, columns=CATEGORICAL_COLS)
    X_test  = pd.get_dummies(X_test_raw,  columns=CATEGORICAL_COLS)

    # Align: test set gets exactly the same columns as train, missing ones filled with 0
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = encode_and_align(train_df, test_df)

print(f"Feature dimensions : {X_train.shape[1]} columns after encoding")
print(f"U2R samples (train): {(y_train == 4).sum()} ({(y_train == 4).sum()/len(y_train)*100:.3f}%)")
print()
X_train.head(3)

Feature dimensions : 122 columns after encoding
U2R samples (train): 52 (0.041%)



,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0,491,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,True,False
1,0,146,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,True,False
2,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False


## Step 4 — Save Processed Splits to `data/processed/`

In [12]:
os.makedirs(OUT_DIR, exist_ok=True)

joblib.dump(X_train, os.path.join(OUT_DIR, 'X_train.pkl'))
joblib.dump(X_test,  os.path.join(OUT_DIR, 'X_test.pkl'))
joblib.dump(y_train, os.path.join(OUT_DIR, 'y_train.pkl'))
joblib.dump(y_test,  os.path.join(OUT_DIR, 'y_test.pkl'))

print("Saved:")
for f in ['X_train.pkl', 'X_test.pkl', 'y_train.pkl', 'y_test.pkl']:
    size = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f"  {OUT_DIR}/{f}  ({size:.1f} KB)")

print("\n✅ Done. Run train_model.ipynb next.")

Saved:
  ..\data\processed/X_train.pkl  (47735.6 KB)
  ..\data\processed/X_test.pkl  (9471.0 KB)
  ..\data\processed/y_train.pkl  (1969.1 KB)
  ..\data\processed/y_test.pkl  (353.0 KB)

✅ Done. Run train_model.ipynb next.
